# المحاضرة 37: أساسيات وخوارزمية التقطيع الدلالي المتقدم
## القسم الأول: الشرح النظري والمعماري الشامل

### 1. قصور التقطيع الحجمي الثابت (The Failure of Fixed-Size Chunking)
في المراحل الأولى من أنظمة RAG، يتم الاعتماد عادة على خوارزميات التقطيع الحجمي الثابت مثل `RecursiveCharacterTextSplitter`. ورغم بساطتها وسرعتها ($O(N)$)، إلا أنها تتسبب في عيوب معمارية خطيرة في بيئات الإنتاج:
- **تمزيق الأفكار والسياق (Semantic Mutilation)**: إذا بدأت الفكرة عند الحرف 400 واكتملت عند الحرف 550 ضمن نافذة حجمها 500 حرف، يتم قطع الفكرة لنصفين في متجهين منفصلين، مما يفسد قدرة المسترجع على جلب سياق كامل.
- **تشتت المتجهات وتلوث السياق (Vector Smearing)**: عند دمج موضوعين متباينين في نفس القطعة لمجرد استكمال عدد الحروف المطلوب، يعبر المتجه الكثيف عن متوسط رياضي مشوش لا يمثل أي موضوع بدقة.

---

### 2. فلسفة التقطيع الدلالي (Semantic Chunking Philosophy)
يعتمد التقطيع الدلالي على فصل النصوص استناداً إلى اكتمال الفكرة واستمرار السياق وليس بناءً على عداد الحروف.
تتميز القطع الدلالية بثلاث خصائص هندسية:
1. **مستقلة بذاتها (Self-Contained)**: كل قطعة تمثل فكرة متكاملة يمكن فهمها دون الحاجة لنصوص مجاورة.
2. **عالية التركيز الدلالي (Contextually Dense)**: ترفع نسبة الإشارة إلى التشويش (Signal-to-Noise Ratio) داخل مخزن المتجهات.
3. **مفصولة منطقياً (Logically Separated)**: نقاط الانفصال تقع فقط عند تحول الموضوعات.

---

### 3. خوارزمية التقطيع الدلالي خطوة بخطوة

```mermaid
flowchart TD
    subgraph Phase1 ["1. التجزئة اللغوية (Segmentation)"]
        Raw["نص خام متعدد الموضوعات"] --> Split["تقسيم الجمل عبر Regex
(?<=[.!?])\s+"]
    end

    subgraph Phase2 ["2. حساب التضمينات (Embeddings)"]
        Split --> Model["HuggingFaceEmbeddings
(all-MiniLM-L6-v2)"]
        Model --> Dense["متجهات الجمل E_0, E_1, ..., E_n"]
    end

    subgraph Phase3 ["3. قياس التشابه الزوجي (Cosine Similarity)"]
        Dense --> Sim["حساب التشابه بين كل جملتين متتاليتين
Sim(E_i, E_{i+1})"]
    end

    subgraph Phase4 ["4. رصد نقاط الانفصال والدمج (Breakpoint Grouping)"]
        Sim --> Check{"هل Sim >= Threshold؟"}
        Check -->|نعم| Merge["دمج الجملة في القطعة الحالية"]
        Check -->|لا| Cut["نقطة انفصال: قفل القطعة وبدء قطعة جديدة"]
    end
```

---

### 4. التحليل الرياضي ومقارنة التقطيع
- يتم قياس التشابه بين متجهات الجمل المتتالية باستخدام جيب التمام:
  $$\text{Cosine Similarity}(u, v) = \frac{u \cdot v}{\|u\| \|v\|}$$
- حدوث انخفاض مفاجئ (Valley / Drop) يشير رياضياً إلى انتقال الكاتب لموضوع غير مرتبط، وهو المكان الدقيق لوضع حد القطعة.


## القسم الثاني: التطبيق البرمجي المتكامل
الكود البرمجي الكامل لتوضيح قصور التقطيع الحجمي الثابت، وتنفيذ خوارزمية التقطيع الدلالي وحساب نقاط الانفصال برمجياً وبيانياً.

In [1]:
# 1. استدعاء المكتبات وتهيئة البيئة
import re
import warnings
import numpy as np
import matplotlib
matplotlib.use("Agg")  # Headless backend to prevent GUI blocking
import matplotlib.pyplot as plt

warnings.filterwarnings("ignore")

from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_huggingface import HuggingFaceEmbeddings

# Initialize local embedding model
embeddings_model = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")

print("[INFO] Libraries loaded successfully.")
print("[INFO] Embedding model: sentence-transformers/all-MiniLM-L6-v2")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

[INFO] Libraries loaded successfully.
[INFO] Embedding model: sentence-transformers/all-MiniLM-L6-v2


In [2]:
# 2. النص النموذجي وعرض مشكلة التقطيع الحجمي الثابت
sample_text = (
    "LangChain is a framework to build RAG applications. "
    "LangChain can also be used to build Generative AI applications and multi-agent workflows. "
    "Paris is the capital of France. "
    "The Eiffel Tower is located in Paris, France and is a popular destination."
)

print(f"[CORPUS] Total characters: {len(sample_text)}\n")

# Apply fixed-size chunking with small chunk size to demonstrate issue
fixed_splitter = RecursiveCharacterTextSplitter(chunk_size=115, chunk_overlap=15)
fixed_chunks = fixed_splitter.split_text(sample_text)

print(f"[FIXED-SIZE SPLITTER] Produced {len(fixed_chunks)} chunks:\n")
for idx, chunk in enumerate(fixed_chunks, 1):
    print(f"--- Fixed Chunk [{idx}] ({len(chunk)} chars) ---")
    print(f"'{chunk}'\n")

[CORPUS] Total characters: 248

[FIXED-SIZE SPLITTER] Produced 3 chunks:

--- Fixed Chunk [1] (114 chars) ---
'LangChain is a framework to build RAG applications. LangChain can also be used to build Generative AI applications'

--- Fixed Chunk [2] (109 chars) ---
'applications and multi-agent workflows. Paris is the capital of France. The Eiffel Tower is located in Paris,'

--- Fixed Chunk [3] (46 chars) ---
'in Paris, France and is a popular destination.'



In [3]:
# 3. خوارزمية التقطيع الدلالي: تجزئة الجمل وتوليد المتجهات
# Step 1: Segmentation into sentences using regex boundary detection
sentences = [s.strip() for s in re.split(r"(?<=[.!?])\s+", sample_text) if s.strip()]

print(f"[SEMANTIC SPLITTER] Extracted {len(sentences)} individual sentences:\n")
for i, s in enumerate(sentences, 1):
    print(f"[{i}] {s}")

# Step 2: Dense embeddings generation
sentence_vectors = np.array(embeddings_model.embed_documents(sentences))
print(f"\n[INFO] Generated embeddings shape: {sentence_vectors.shape}")

[SEMANTIC SPLITTER] Extracted 4 individual sentences:

[1] LangChain is a framework to build RAG applications.
[2] LangChain can also be used to build Generative AI applications and multi-agent workflows.
[3] Paris is the capital of France.
[4] The Eiffel Tower is located in Paris, France and is a popular destination.

[INFO] Generated embeddings shape: (4, 384)


In [4]:
# 4. قياس التشابه الزوجي ورصد الهبوط الحاد (Pairwise Similarity)
def cosine_sim(u, v):
    norm_u = np.linalg.norm(u)
    norm_v = np.linalg.norm(v)
    if norm_u == 0 or norm_v == 0:
        return 0.0
    return float(np.dot(u, v) / (norm_u * norm_v))

similarities = []
print("[INFO] Pairwise Adjacent Similarities:\n")
for i in range(len(sentence_vectors) - 1):
    sim = cosine_sim(sentence_vectors[i], sentence_vectors[i + 1])
    similarities.append(sim)
    print(f"Similarity (S_{i+1} -> S_{i+2}): {sim:.4f}")
    print(f"   S_{i+1}: '{sentences[i][:40]}...'")
    print(f"   S_{i+2}: '{sentences[i+1][:40]}...'")
    if sim < 0.2:
        print("   [BREAKPOINT DETECTED] Severe semantic drop (topic transition).\n")
    else:
        print("   [CONTINUITY] High semantic continuity.\n")

[INFO] Pairwise Adjacent Similarities:

Similarity (S_1 -> S_2): 0.5493
   S_1: 'LangChain is a framework to build RAG ap...'
   S_2: 'LangChain can also be used to build Gene...'
   [CONTINUITY] High semantic continuity.

Similarity (S_2 -> S_3): -0.0077
   S_2: 'LangChain can also be used to build Gene...'
   S_3: 'Paris is the capital of France....'
   [BREAKPOINT DETECTED] Severe semantic drop (topic transition).

Similarity (S_3 -> S_4): 0.5496
   S_3: 'Paris is the capital of France....'
   S_4: 'The Eiffel Tower is located in Paris, Fr...'
   [CONTINUITY] High semantic continuity.



In [5]:
# 5. التجميع الدلالي بناءً على العتبة (Threshold Grouping)
threshold = 0.35  # Threshold clearly distinguishing AI vs Geography

semantic_chunks = []
current_chunk = [sentences[0]]

for i, sim in enumerate(similarities):
    if sim >= threshold:
        current_chunk.append(sentences[i + 1])
    else:
        semantic_chunks.append(" ".join(current_chunk))
        current_chunk = [sentences[i + 1]]

if current_chunk:
    semantic_chunks.append(" ".join(current_chunk))

print(f"[INFO] Semantic Chunker produced {len(semantic_chunks)} cohesive chunks:\n")
for idx, chunk in enumerate(semantic_chunks, 1):
    print(f"--- Semantic Chunk [{idx}] ({len(chunk)} chars) ---")
    print(f"{chunk}\n")

[INFO] Semantic Chunker produced 2 cohesive chunks:

--- Semantic Chunk [1] (141 chars) ---
LangChain is a framework to build RAG applications. LangChain can also be used to build Generative AI applications and multi-agent workflows.

--- Semantic Chunk [2] (106 chars) ---
Paris is the capital of France. The Eiffel Tower is located in Paris, France and is a popular destination.



In [6]:
# 6. الرسم البياني للتشابه ونقاط الانفصال الدلالي
fig, ax = plt.subplots(figsize=(8, 4))
pairs = [f"S{i+1}-S{i+2}" for i in range(len(similarities))]
ax.plot(pairs, similarities, marker="o", color="#1f77b4", linewidth=2, label="Cosine Similarity")
ax.axhline(y=threshold, color="r", linestyle="--", label=f"Threshold ({threshold})")
ax.set_title("Pairwise Sentence Semantic Similarity & Breakpoint", fontsize=12)
ax.set_ylabel("Cosine Similarity", fontsize=10)
ax.set_ylim(0, 1)
ax.grid(True, linestyle=":", alpha=0.6)
ax.legend()
plt.tight_layout()
plt.savefig("semantic_similarity_plot.png", dpi=150)
plt.close()

print("[INFO] Semantic similarity plot saved as 'semantic_similarity_plot.png'.")

[INFO] Semantic similarity plot saved as 'semantic_similarity_plot.png'.


In [7]:
# 7. التحقق البرمجي النهائي (Assertions)
assert len(semantic_chunks) == 2, "Semantic chunker with threshold 0.35 must yield exactly 2 chunks"
assert "Paris" not in semantic_chunks[0], "First chunk must not contain Paris"
assert "LangChain" not in semantic_chunks[1], "Second chunk must not contain LangChain"

print("[SUCCESS] All architectural assertions passed successfully.")

[SUCCESS] All architectural assertions passed successfully.


---
## القسم الثالث: مساحة التجربة والتطبيق العملي
استخدم الخلية التالية لتجربة نصوص مختلفة وملاحظة كيف تختلف نتائج التقطيع الدلالي مقارنة بالتقطيع الحجمي.

In [8]:
# مساحة التجربة العملية:
# جرب إدخال نص يحتوي على مواضيع متعددة واختبر حساب التشابه وتحديد العتبة بنفسك هنا:

